To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth your local device, follow [our guide](https://docs.unsloth.ai/get-started/install-and-update). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News


Unsloth's [Docker image](https://hub.docker.com/r/unsloth/unsloth) is here! Start training with no setup & environment issues. [Read our Guide](https://docs.unsloth.ai/new/how-to-train-llms-with-unsloth-and-docker).

[gpt-oss RL](https://docs.unsloth.ai/new/gpt-oss-reinforcement-learning) is now supported with the fastest inference & lowest VRAM. Try our [new notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/gpt-oss-(20B)-GRPO.ipynb) which creates kernels!

Introducing [Vision](https://docs.unsloth.ai/new/vision-reinforcement-learning-vlm-rl) and [Standby](https://docs.unsloth.ai/basics/memory-efficient-rl) for RL! Train Qwen, Gemma etc. VLMs with GSPO - even faster with less VRAM.

Unsloth now supports Text-to-Speech (TTS) models. Read our [guide here](https://docs.unsloth.ai/basics/text-to-speech-tts-fine-tuning).

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except: get_numpy = "numpy"; get_pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} {get_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@05b2c186c1b6c9a08375389d5efe9cb4c401c075#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [ ]:
%%capture
!uv pip install --force-reinstall --no-deps git+https://github.com/unslothai/unsloth-zoo
!uv pip install --force-reinstall --no-deps git+https://github.com/unslothai/unsloth

### Unsloth

We're about to demonstrate the power of the new OpenAI GPT-OSS 20B model through an inference example. For our `MXFP4` version, use this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/GPT_OSS_MXFP4_(20B)-Inference.ipynb) instead.

In [ ]:
from unsloth import FastLanguageModel
import torch

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/gpt-oss-20b-unsloth-bnb-4bit", # 20B model using bitsandbytes 4bit quantization
    "unsloth/gpt-oss-120b-unsloth-bnb-4bit",
    "unsloth/gpt-oss-20b", # 20B model using MXFP4 format
    "unsloth/gpt-oss-120b",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b-unsloth-bnb-4bit",
    dtype = None, # None for auto detection
    max_seq_length = 4096, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.12: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00000-of-00002.safetensors:   0%|          | 0.00/4.79G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.80G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

### Reasoning Effort
The `gpt-oss` models from OpenAI include a feature that allows users to adjust the model's "reasoning effort." This gives you control over the trade-off between the model's performance and its response speed (latency) which by the amount of token the model will use to think.

----

The `gpt-oss` models offer three distinct levels of reasoning effort you can choose from:

* **Low**: Optimized for tasks that need very fast responses and don't require complex, multi-step reasoning.
* **Medium**: A balance between performance and speed.
* **High**: Provides the strongest reasoning performance for tasks that require it, though this results in higher latency.

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "low", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 512, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-11-01

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>We need to solve equation: x^5 +3x^4 -10 =3 => x^5 +3x^4 -13 =0. Need real solutions. Try integer roots: factors of 13: ±1,±13. Plug x=1:1+3-13=-9. x=13 huge. x=-1: -1+3-13=-11. x= -13 huge. So no integer roots. Use numeric? Maybe there's rational like 13^... not. Could have one real root between 1 and2? At x=1: -9; at x=2:32+48-13=67>0. So root near 1. Probably only one real root. Use approximate. Let's approximate: f(1.2)=1.2^5+3*1.2^4-13. 1.2^4=2.0736, *3=6.2208; 1.2^5=2.4883; sum=8.7089-13=-4.2911. f(1.4):1.4^4=3.8416*3=11.5248;1.4^5=5.3782 sum=16.903-13=3.903. f(1.3):1.3^4=2.8561*3=8.5683;1.3^5=3.7129 sum=12.2812-13=-0.7188. f(1.32)

Changing the `reasoning_effort` to `medium` will make the model think longer. We have to increase the `max_new_tokens` to occupy the amount of the generated tokens but it will give better and more correct answer

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "medium", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 1024, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-11-01

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>The problem: Solve x^5 + 3x^4 - 10 = 3. Presumably solve for real x? Equation: x^5 + 3x^4 - 10 = 3. That simplifies to x^5 + 3x^4 -13 = 0. Solve for x. It's a quintic. There might be a rational root: Try integer candidates dividing 13: ±1, ±13. Test x=1: 1+3-13 = -9 ≠ 0. x=-1: -1 + 3(-1)^4? Wait -1^5 = -1, x^4 =1, so -1 + 3*1 -13 = -11. x=13: 13^5 huge. x=-13: huge. So no integer. Maybe something like x = sqrt(??). But maybe the problem intended x^5 + 3x^4 = 13, solve approximate. Let's find approximate real root(s). f(x)=x^5+3x^4-13. f(1)=1+3-13=-9. f(2)=32+48-13=67>0. So root between 1 and 2. f(1.5) = 1.5^5 + 3*1.5^4 -13. 1.5^2=2.25

Lastly we will test it using `reasoning_effort` to `high`

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "high", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 2048, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-11-01

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>We have an equation: x^5 + 3x^4 - 10 = 3. So x^5 + 3x^4 - 13 = 0. We need to solve for x. This is a quintic. Typically, general quintic is unsolvable by radicals. But maybe it's simple integer root? Let's test integers: x=1: 1+3-13 = -9, not zero. x=2: 32 + 48 - 13 = 67, too high. x=-1: -1 + 3 - 13 = -11. x= -2: -32 + 48 - 13 = 3, not zero. x= -3: -243 + 243 - 13 = -13. x= 3: 243 + 3*81 - 13 = 243 + 243 - 13 = 473, too high. So maybe there is rational root? Rational roots are factors of constant term 13 over leading coefficient 1. ±1, ±13. We tested 1 and -1. 13: huge. -13: negative huge. So no rational root. It might be some real root 

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)
</div>


# 🧠 GPT-OSS Reasoning over Financial News (Explainers per Row)

This section adds a **dataset-driven reasoning** pipeline on top of the existing GPT-OSS (20B) inference setup.
Given a CSV of news items with gold sentiment labels, the model will:
- read each row,
- **explain why** the item is Neutral/Positive/Negative,
- output a **concise justification** and a **predicted label**,
- log results into a dataframe and CSV for review.

_Last updated: 2025-11-02 22:26 UTC_

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

# --- 1) Path to your CSV dataset ---
CSV_PATH = "/content/drive/MyDrive/SentimentAnalysisforFinancialNews-subset.csv"  # change if needed
# DATA_PATH = "/content/drive/MyDrive/SentimentAnalysisforFinancialNews.csv"

# --- 2) Read dataset (two columns: label, text) ---
import pandas as pd
import numpy as np

# Try common encodings
_enc_try = ["utf-8", "utf-8-sig", "cp1252", "iso-8859-1"]
_df = None
for _enc in _enc_try:
    try:
        _df = pd.read_csv(CSV_PATH, encoding=_enc)
        break
    except Exception as e:
        pass
if _df is None:
    raise RuntimeError("Could not read CSV. Try changing CSV_PATH or encoding.")

# Expect first column = gold label, second column = text
if _df.shape[1] < 2:
    raise ValueError("Expected at least 2 columns: [label, text]. Found: %s" % list(_df.columns))

# Normalise column names
_df = _df.rename(columns={_df.columns[0]: "gold_label", _df.columns[1]: "text"}).copy()
_df = _df.dropna(subset=["text"]).reset_index(drop=True)

print("Loaded rows:", len(_df))
print("Columns:", list(_df.columns))
print(_df.head(3))


Mounted at /content/drive
Loaded rows: 101
Columns: ['gold_label', 'text']
  gold_label                                               text
0    neutral  According to Gran , the company has no plans t...
1    neutral  Technopolis plans to develop in stages an area...
2   negative  The international electronic industry company ...


In [ ]:

# --- 3) Reasoning helper using the already loaded `model` and `tokenizer` ---
from transformers import TextStreamer
import torch, json, re, math
from tqdm.auto import tqdm

def build_reasoning_prompt(text, gold_label):
    # Keep explanation SHORT and CONCRETE to avoid rambling
    system = (
        "You are a financial NLP assistant. "
        "Classify the sentiment of the provided news as Positive, Negative, or Neutral. "
        "Then give a SHORT, concrete explanation citing the phrases that led you there."
    )
    user = f"""News:
{text}

Gold label: {gold_label}

Instructions:
1) Decide the sentiment strictly from the news content.
2) Keep the explanation to 1-3 sentences, focusing on evidence (numbers, direction, outcomes).
3) Output JSON with keys: predicted_label, explanation.
"""
    return system, user

def gpt_oss_chat_json(system, user, max_new_tokens=200, temperature=0.2, top_p=0.9):
    # Requires `model` and `tokenizer` to be defined in the runtime, as per the original notebook.
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            eos_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.batch_decode(outputs)[0]
    # Extract model reply after the prompt
    reply = decoded[len(prompt):].strip()
    # Try to find JSON in the reply
    match = re.search(r"\{.*\}", reply, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0)), reply
        except Exception:
            pass
    # Fallback: return as free text
    return {{"predicted_label": None, "explanation": reply[:800]}}, reply

def run_reasoning(df, sample_size=None):
    data = df if sample_size is None else df.sample(sample_size, random_state=42).reset_index(drop=True)
    records = []
    for i, row in tqdm(data.iterrows(), total=len(data)):
        gold = str(row["gold_label"]).strip().lower()
        text = str(row["text"]).strip()
        sys, usr = build_reasoning_prompt(text, gold)
        js, raw = gpt_oss_chat_json(sys, usr)
        pred = (js.get("predicted_label") or "").strip().lower()
        exp = js.get("explanation") or ""
        records.append({
            "text": text,
            "gold_label": gold,
            "predicted_label": pred,
            "explanation": exp,
            "raw_model_output": raw,
        })
    return pd.DataFrame(records)

print("Reasoning helpers ready. Ensure `model` & `tokenizer` are loaded above (per original cells).")


Reasoning helpers ready. Ensure `model` & `tokenizer` are loaded above (per original cells).


In [ ]:

# --- 4) Run on the whole dataset or a sample ---
# Tip: start with a small sample to validate formatting, then scale.
SAMPLE = 10   # set to None for full dataset
results_df = run_reasoning(_df, sample_size=SAMPLE)

# Simple accuracy if model outputs labels properly
def normalise_label(x):
    x = (x or "").lower()
    if "pos" in x: return "positive"
    if "neg" in x: return "negative"
    if "neu" in x: return "neutral"
    return x

results_df["predicted_label_norm"] = results_df["predicted_label"].map(normalise_label)
results_df["gold_label_norm"] = results_df["gold_label"].map(normalise_label)
acc = (results_df["predicted_label_norm"] == results_df["gold_label_norm"]).mean()

print(f"Sample size: {len(results_df)}")
print(f"Exact label match (normalised): {acc:.3f}")
results_df.head(5)


  0%|          | 0/10 [00:00<?, ?it/s]

Sample size: 10
Exact label match (normalised): 1.000


,text,gold_label,predicted_label,explanation,raw_model_output,predicted_label_norm,gold_label_norm
0,The combination of all services enabling us to...,positive,positive,"The statement highlights an expanded, strength...",<|channel|>analysis<|message|>We need to class...,positive,positive
1,Shares of Nokia Corp. rose Thursday after the ...,positive,positive,The news reports that Nokia’s third‑quarter ea...,<|channel|>analysis<|message|>We need to outpu...,positive,positive
2,Sponda Plc Stock Exchange Release 5 December 2...,positive,positive,The announcement of a EUR 1.5 billion syndicat...,<|channel|>analysis<|message|>We need to class...,positive,positive
3,STOCK EXCHANGE ANNOUNCEMENT 20 July 2006 1 ( 1...,neutral,neutral,The announcement reports a subscription of 119...,<|channel|>analysis<|message|>We need to class...,neutral,neutral
4,Sales increased due to growing market rates an...,positive,positive,"The news states ""Sales increased"" and attribut...",<|channel|>analysis<|message|>We need to outpu...,positive,positive


In [ ]:

# --- 5) Save results for review ---
OUT_CSV = "/content/drive/MyDrive/gpt_oss_reasoning_outputs.csv"
results_df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)


Saved: /content/drive/MyDrive/gpt_oss_reasoning_outputs.csv


In [ ]:

# --- (Optional) 6) Inspect a single row interactively ---
ROW_INDEX = 0  # change this
row = _df.iloc[ROW_INDEX]
sys, usr = build_reasoning_prompt(str(row['text']), str(row['gold_label']))
js, raw = gpt_oss_chat_json(sys, usr, max_new_tokens=200)
print("GOLD:", row['gold_label'])
print("MODEL:", js.get("predicted_label"))
print("EXPLANATION:", js.get("explanation"))


GOLD: neutral
MODEL: neutral
EXPLANATION: The article states the company has no plans to relocate all production to Russia, yet it is growing there—no clear positive or negative outcome is highlighted.
